# Correction methods of spectral band in EEG synthetic data

In [7]:
%load_ext autoreload
%autoreload 2
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from Functions.generate_OU import get_mixed_OU_signals
from Functions.time_frequency import spectrogram
import pywt
import cv2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Generate synthetic signal

We generate a synthetic signal. In the EEG we want to mimic the presence of $\delta$ + $\alpha$ bands, as it would be for a gabaergic anesthesia and the addition of a $\beta$ band to mimic the __ketamine signature__.

To generate one band we use a __2D Ornstein-Ulhenbeck__ process that can create spindles like patterns with waning and waxing envelopes containing oscillations.

In [8]:
# --- Parameters
T = 20 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2, 1]
omega_list = [2*np.pi*1, 2*np.pi*10, 2*np.pi*30]
sigma_list = [3, 2, 2]
factor_list = [1, 1, 1]

# --- Generate EEG data
t, y = get_mixed_OU_signals(T, dt, lbda_list, omega_list, sigma_list, factor_list)

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs)
mask_f = f_spectro >= 20
f_spectro = f_spectro[mask_f]
spectro = spectro[mask_f, :]

# --- Display
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spectro, np.log2(np.sum(spectro, axis = 1)))
axes[2].set_title('PSD')
axes[1].sharex(axes[0])

plt.show()

ValueError: too many values to unpack (expected 2)

## 2. Transform of the data

We perform various transforms on the data to visualize which one emphasize more the horizontal band in $\beta$ regime. We can also swap easily the matrix input from a transform to check its effect on the method efficiency. 

In [ ]:
M = np.log2(spectro + 1e-11) # standard input

# --- 1. Base Pipeline ---
# M_gauss: M filtered through a 5x5 Gaussian kernel
M_gauss = cv2.GaussianBlur(M, (5, 5), sigmaX=0)

# M_grad: Gradient magnitude of original M using 5x5 Sobel
M_grad = np.abs(cv2.Sobel(M, cv2.CV_64F, 1, 1, ksize=5))

# M_grad_gauss: M_grad filtered through a 5x5 Gaussian filter
M_grad_gauss = cv2.GaussianBlur(M_grad, (5, 5), sigmaX=0)

# --- 2. New Pipeline (Gradient of Gauss) ---
# M_gauss_grad: Gradient magnitude computed FROM the blurred image M_gauss
M_gauss_grad = np.abs(cv2.Sobel(M_gauss, cv2.CV_64F, 1, 1, ksize=5))

# M_gauss_grad_gauss: The blurred version of the gradient of gauss
M_gauss_grad_gauss = cv2.GaussianBlur(M_gauss_grad, (5, 5), sigmaX=0)


# --- 3. Visualization (Expanded to 2x3 Grid) ---
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# --- Row 1: Operations starting from original M ---
axes[0, 0].pcolormesh(M, shading='nearest', cmap='jet')
axes[0, 0].set_title("Original M")

axes[0, 1].pcolormesh(M_grad, shading='nearest', cmap='jet')
axes[0, 1].set_title("M_grad (Gradient of M)")

axes[0, 2].pcolormesh(M_grad_gauss, shading='nearest', cmap='jet')
axes[0, 2].set_title("M_grad_gauss (Smoothed)")

# --- Row 2: Operations starting from blurred M_gauss ---
axes[1, 0].pcolormesh(M_gauss, shading='nearest', cmap='jet')
axes[1, 1].set_title("M_gauss (Blurred)")

axes[1, 1].pcolormesh(M_gauss_grad, shading='nearest', cmap='jet')
axes[1, 1].set_title("M_gauss_grad (Gradient of Gauss)")

axes[1, 2].pcolormesh(M_gauss_grad_gauss, shading='nearest', cmap='jet')
axes[1, 2].set_title("M_gauss_grad_gauss (Smoothed)")


# --- Axis Formatting ---
for ax in axes.ravel():
    ax.invert_yaxis()    # Corrects vertical orientation for spectrogram data
    ax.set_aspect('auto') # Allows the spectrogram time/freq dimensions to scale naturally
    ax.axis('off')       # Hides background grid lines

plt.tight_layout()
plt.show()

## 3. Wavelet decomposition

The discrete wavelet decomposition in 2D can be use to emphasize spoecific patterns in the spectrogram image like horizontal bands. We start to analyise the effect of a 1 level decomposition below.

In [ ]:
coeffs = pywt.dwt2(M, 'db1')
cA, (cH, cV, cD) = coeffs

n_f, n_t = np.shape(cA) # shape of 2D DWT coefficients
t_wav = np.linspace(t_spectro[0], t_spectro[-1], n_t)
f_wav = np.linspace(f_spectro[0], f_spectro[-1], n_f)


fig, axes = plt.subplots(2, 3, sharex = True, constrained_layout = True)
axes[0, 0].plot(t, y)
axes[0, 0].set_title('Simulated EEG signal')
axes[1, 0].pcolormesh(t_spectro, f_spectro, M, shading = 'nearest', cmap = 'jet')
axes[1, 0].set_title('Spectrogram')

axes[0, 1].pcolormesh(t_wav, f_wav, cA, shading = 'nearest', cmap = 'jet')
axes[0, 1].set_title('cA coeff')
axes[1, 1].pcolormesh(t_wav, f_wav, cH, shading = 'nearest', cmap = 'jet')
axes[1, 1].set_title('cH coeff')

axes[0, 2].pcolormesh(t_wav, f_wav, cV, shading = 'nearest', cmap = 'jet')
axes[0, 2].set_title('cV coeff')
axes[1, 2].pcolormesh(t_wav, f_wav, cD, shading = 'nearest', cmap = 'jet')
axes[1, 2].set_title('cD coeff')

plt.show()


To investigate further we switch to a 3 level decomposition.

In [ ]:
# Define the number of decomposition levels you want
levels = 3

# 1. Multi-level 2D Discrete Wavelet Decomposition
# coeffs list structure: [cA_n, (cH_n, cV_n, cD_n), ..., (cH_1, cV_1, cD_1)]
coeffs = pywt.wavedec2(M_gauss, 'db1', level=levels)

# Extract the main approximation coefficient (deepest level)
cA = coeffs[0]

# --- Plotting Setup ---
# We will create a grid where:
# Row 0 handles the original Signal and Spectrogram
# Rows 1 to 'levels' will display cA, cH, cV, and cD for each level
fig, axes = plt.subplots(levels + 1, 4, figsize=(14, 3 * (levels + 1)), constrained_layout=True)

# Turn off any unused axes in the top row to keep it clean
for col in range(2, 4):
    axes[0, col].axis('off')

# 2. Plot Original Signal & Spectrogram (Top Row)
axes[0, 0].plot(t, y)
axes[0, 0].set_title('Simulated EEG signal')

axes[0, 1].pcolormesh(t_spectro, f_spectro, M, shading='nearest', cmap='jet')
axes[0, 1].set_title('Original Spectrogram (M)')

# 3. Dynamically Plot Coefficients for Each Level
for i in range(levels):
    # Map the coefficients correctly from the wavedec2 output list
    # Level 3 is index 1, Level 2 is index 2, Level 1 is index 3
    cH, cV, cD = coeffs[i + 1]
    
    # Determine current level label (e.g., Level 3 down to Level 1)
    current_level = levels - i 
    
    # Find the shape for the current level's coefficient matrix
    n_f, n_t = np.shape(cH)
    t_wav = np.linspace(t_spectro[0], t_spectro[-1], n_t)
    f_wav = np.linspace(f_spectro[0], f_spectro[-1], n_f)
    
    # Row index in our plot grid for this level
    row = i + 1
    
    # Plot cA only for the deepest level grid slot (or keep it aligned per row)
    # Note: wavedec2 only produces ONE final cA at the deepest level (Level 3 here)
    if i == 0:
        n_f_cA, n_t_cA = np.shape(cA)
        t_wav_cA = np.linspace(t_spectro[0], t_spectro[-1], n_t_cA)
        f_wav_cA = np.linspace(f_spectro[0], f_spectro[-1], n_f_cA)
        axes[row, 0].pcolormesh(t_wav_cA, f_wav_cA, cA, shading='nearest', cmap='jet')
        axes[row, 0].set_title(f'cA{current_level} (Approximation)')
    else:
        # Hide the cA column for subsequent rows since it's only calculated once at the bottom
        axes[row, 0].axis('off')
        axes[row, 0].set_title(f'cA{current_level} (N/A)')

    # Plot details: Horizontal, Vertical, Diagonal
    axes[row, 1].pcolormesh(t_wav, f_wav, cH, shading='nearest', cmap='jet')
    axes[row, 1].set_title(f'cH{current_level} (Horizontal Detail)')
    
    axes[row, 2].pcolormesh(t_wav, f_wav, cV, shading='nearest', cmap='jet')
    axes[row, 2].set_title(f'cV{current_level} (Vertical Detail)')
    
    axes[row, 3].pcolormesh(t_wav, f_wav, cD, shading='nearest', cmap='jet')
    axes[row, 3].set_title(f'cD{current_level} (Diagonal Detail)')

plt.show()

## 4. Adaptive thresholding

## Segmentation by PSD baseline

On the PSD, considering a method of detection of the beta bump is already provided, one can fit a baseline to this bump. Now any data higher than the corresponding baseline at a given frequency can be considered part of a burst of activity. Now they should be a part either including the fact that its close neigbour and pixel form an overall blob higher than psd. And/or include a flexibility of the threshold. 